# V-JEPA 2 predictor check (Kaggle, self-contained)

Does the predictor ("world model") help tracking? `scripts/vjepa_predictor_check.py`:
- **A** surprise of the predictor at the target's last place as an occlusion detector (AUROC)
- **B** predicted future frame: is the target where the predictor says?

40 dev videos, every 16th frame, ~45-60 min on T4 x2. Output: `/kaggle/working/predictor_check.zip`.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, no inputs. Then **Save Version -> Save & Run All**.

In [ ]:
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On, then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q "transformers>=4.53"

In [ ]:
import os, subprocess, json, torch
os.chdir('/kaggle/working/Object-Tracking')
os.environ['HF_HUB_DISABLE_XET'] = '1'
W, OUT = '/kaggle/working', '/kaggle/working/results/vjepa_predictor_check'
DEV = 'configs/splits/dev_occlusion.txt'
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 2 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)

## Self-test: one video, 100 frames (fails within minutes if something is broken)

In [ ]:
subprocess.run(['python', 'scripts/vjepa_predictor_check.py', '--sequences', 'airplane-1', '--max-frames', '100',
                '--out', '/kaggle/tmp/pred_selftest', '--work', '/kaggle/tmp'], check=True)
print('SELF-TEST OK')

## Run on the 40 dev videos

In [ ]:
for attempt in range(2):  # a re-run only does the videos that failed to download
    run_shards('predictor', ['scripts/vjepa_predictor_check.py', '--sequences', DEV, '--out', OUT, '--work', '/kaggle/tmp'])
# summary over all shards
subprocess.run(['python', 'scripts/vjepa_predictor_check.py', '--sequences', DEV, '--out', OUT], check=True)
os.system(f'cd {W} && rm -f predictor_check.zip && zip -qr predictor_check.zip results logs; ls -la {W}/predictor_check.zip')
os.system(f'rm -rf {W}/Object-Tracking')